In [1]:
import numpy as np

def p(j, i, Nx):
    return j * Nx + i

def build_full_node_matrix(Nx_steps, Ny_steps):
    """
    Build the FULL A matrix (including boundary nodes), matching the C++ code.
    
    Nx_steps, Ny_steps = number of steps (not nodes)
    Total nodes in x = Nx_steps + 1
    Total nodes in y = Ny_steps + 1
    """

    Nx = Nx_steps + 1
    Ny = Ny_steps + 1
    N = Nx * Ny

    A = np.zeros((N, N))
    b = np.zeros(N)

    dx = 1.0 / Nx_steps
    dy = 1.0 / Ny_steps
    cx = 1.0 / dx**2
    cy = 1.0 / dy**2

    # Loop over all nodes INCLUDING boundaries
    for j in range(Ny):
        for i in range(Nx):

            row = p(j, i, Nx)

            # BOUNDARIES -------------------------------------------------------
            if i == 0:                      # left boundary
                A[row, row] = 1.0
                b[row] = 0.0
                continue

            if i == Nx - 1:                # right boundary
                A[row, row] = 1.0
                b[row] = 0.0
                continue

            if j == 0:                     # bottom boundary
                A[row, row] = 1.0
                b[row] = 0.0
                continue

            if j == Ny - 1:                # top boundary
                A[row, row] = 1.0
                b[row] = 1.0
                continue

            # INTERIOR ----------------------------------------------------------
            A[row, row] = -2.0 * (cx + cy)
            A[row, p(j, i - 1, Nx)] = cx
            A[row, p(j, i + 1, Nx)] = cx
            A[row, p(j - 1, i, Nx)] = cy
            A[row, p(j + 1, i, Nx)] = cy

    return A, b


# =====================================================================
# JACOBI + SPECTRAL RADIUS + OPTIMAL OMEGA
# =====================================================================

def jacobi_iteration_matrix(A):
    """Jacobi iteration matrix J = -D^{-1}(L + U)."""
    D = np.diag(np.diag(A))
    R = A - D
    return -np.linalg.inv(D) @ R

def spectral_radius(M):
    eigs = np.linalg.eigvals(M)
    return np.max(np.abs(eigs))

def compute_optimal_omega(A):
    J = jacobi_iteration_matrix(A)
    rho = spectral_radius(J)

    w_opt = 2.0 / (1.0 + np.sqrt(1 - rho*rho))
    return w_opt, rho


# =====================================================================
# Example Run
# =====================================================================

if __name__ == "__main__":
    Nx_steps = 119
    Ny_steps = 119

    A, b = build_full_node_matrix(Nx_steps, Ny_steps)

    print("Matrix size =", A.shape)

    omega_opt, rhoJ = compute_optimal_omega(A)

    print("Spectral radius ρ(J) =", rhoJ)
    print("Optimal SOR ω =", omega_opt)


Matrix size = (14400, 14400)


: 

In [5]:
import numpy as np

def optimal_sor_omega(Nx_steps, Ny_steps):
    """
    Compute optimal SOR omega for 2D Poisson problem on a unit square
    using full-node discretization (matching your C++ code).
    
    Nx_steps, Ny_steps : number of intervals (steps)
    """

    Nx = Nx_steps     # interior intervals in x
    Ny = Ny_steps     # interior intervals in y

    # Spectral radius of Jacobi iteration matrix (analytic)
    rhoJ = 0.5 * (np.cos(np.pi / Nx) + np.cos(np.pi / Ny))

    # Optimal SOR relaxation factor
    omega_opt = 2.0 / (1.0 + np.sqrt(1.0 - rhoJ * rhoJ))

    return omega_opt, rhoJ


if __name__ == "__main__":
    Nx_steps = 19
    Ny_steps = 19

    omega, rho = optimal_sor_omega(Nx_steps, Ny_steps)

    print("Spectral radius ρ(J) =", rho)
    print("Optimal SOR ω =", omega)


Spectral radius ρ(J) = 0.9863613034027223
Optimal SOR ω = 1.7173358151336466
